<a href="https://colab.research.google.com/github/bigbathtub101/sector-rotation-system/blob/main/notebooks/setup_and_backtest.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Setup & Backtest Notebook

This notebook loads FRED sector indices + macro data, builds a simple feature set, trains a rolling model (or runs a rules-only baseline), and evaluates performance.

**Repo structure expected** (run from repo root):
- sector_rotation_system/
- notebooks/
- data/ (optional; cached downloads)


In [ ]:
# If running in Colab, mount drive (optional)
# from google.colab import drive
# drive.mount('/content/drive')

In [ ]:
import os
import sys
from pathlib import Path
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')

# Ensure repo root is on path
repo_root = Path('.').resolve()
if (repo_root / 'sector_rotation_system').exists():
    sys.path.append(str(repo_root))
else:
    # If notebook is run from within notebooks/, go up one level
    repo_root = Path('..').resolve()
    sys.path.append(str(repo_root))

print('Repo root:', repo_root)
print('Python:', sys.version)

## 1) Configuration

In [ ]:
# Configuration
DATA_DIR = repo_root / 'data'
DATA_DIR.mkdir(exist_ok=True)

START_DATE = '1990-01-01'
END_DATE = None  # None => today

# FRED series: choose sector indices and a few macro indicators
SECTOR_SERIES = {
    # Examples; adjust to your actual series IDs
    'SP500': 'SP500',
    'INDPRO': 'INDPRO',
}

MACRO_SERIES = {
    'CPIAUCSL': 'CPIAUCSL',
    'UNRATE': 'UNRATE',
    'FEDFUNDS': 'FEDFUNDS',
}

# Backtest settings
REBAlANCE_FREQ = 'M'
LOOKBACK_MONTHS = 12

# Model settings
USE_MODEL = True
ROLLING_TRAIN_WINDOW_MONTHS = 60

RANDOM_STATE = 42

## 2) Data download helpers (FRED)

In [ ]:
import requests

def fred_download_series(series_id: str, api_key: str = None, start_date: str = None, end_date: str = None) -> pd.Series:
    """Download a FRED series as a pandas Series."""
    base = 'https://api.stlouisfed.org/fred/series/observations'
    params = {
        'series_id': series_id,
        'api_key': api_key or os.getenv('FRED_API_KEY', ''),
        'file_type': 'json',
    }
    if start_date:
        params['observation_start'] = start_date
    if end_date:
        params['observation_end'] = end_date

    r = requests.get(base, params=params)
    r.raise_for_status()
    j = r.json()
    obs = j.get('observations', [])
    dates = [o['date'] for o in obs]
    vals = [float(o['value']) if o['value'] not in ('.', None, '') else np.nan for o in obs]
    s = pd.Series(vals, index=pd.to_datetime(dates), name=series_id).sort_index()
    return s

def cache_series(series_id: str, s: pd.Series) -> Path:
    path = DATA_DIR / f'{series_id}.csv'
    s.to_csv(path, header=True)
    return path

def load_cached_series(series_id: str) -> pd.Series | None:
    path = DATA_DIR / f'{series_id}.csv'
    if path.exists():
        df = pd.read_csv(path, index_col=0, parse_dates=True)
        return df.iloc[:, 0].rename(series_id)
    return None

## 3) Load data (from cache if available)

In [ ]:
def get_series(series_id: str) -> pd.Series:
    cached = load_cached_series(series_id)
    if cached is not None:
        return cached
    s = fred_download_series(series_id, start_date=START_DATE, end_date=END_DATE)
    cache_series(series_id, s)
    return s

# Load all series
series_data = {}
for name, sid in {**SECTOR_SERIES, **MACRO_SERIES}.items():
    print('Loading', name, sid)
    series_data[name] = get_series(sid)

df = pd.concat(series_data.values(), axis=1)
df.columns = list(series_data.keys())
df = df.sort_index()
df.head()

## 4) Feature engineering

In [ ]:
# Resample monthly and compute returns
monthly = df.resample('M').last()
monthly_rets = monthly.pct_change()

# Simple macro deltas
macro_cols = list(MACRO_SERIES.keys())
macro_delta = monthly[macro_cols].diff()

# Momentum features on sectors
sector_cols = list(SECTOR_SERIES.keys())
mom_3 = monthly[sector_cols].pct_change(3)
mom_6 = monthly[sector_cols].pct_change(6)
mom_12 = monthly[sector_cols].pct_change(12)

features = pd.concat({
    'rets': monthly_rets[sector_cols],
    'mom3': mom_3,
    'mom6': mom_6,
    'mom12': mom_12,
    'macro_delta': macro_delta
}, axis=1)

features = features.dropna()
features.head()

## 5) Backtest helpers

In [ ]:
def compute_portfolio_returns(weights: pd.DataFrame, returns: pd.DataFrame) -> pd.Series:
    # Align and compute portfolio returns
    w = weights.reindex(returns.index).fillna(method='ffill').fillna(0)
    r = returns.reindex(w.index).fillna(0)
    port = (w * r).sum(axis=1)
    return port

def sharpe_ratio(r: pd.Series, periods_per_year=12) -> float:
    if r.std() == 0:
        return np.nan
    return (r.mean() / r.std()) * np.sqrt(periods_per_year)

def max_drawdown(cum: pd.Series) -> float:
    roll_max = cum.cummax()
    dd = (cum / roll_max) - 1
    return dd.min()

## 6) Simple strategy: momentum rotation (baseline)

In [ ]:
# Baseline: pick top sector by 12m momentum each rebalance date
mom = mom_12.dropna()
rebalance_dates = mom.index

weights = pd.DataFrame(0, index=rebalance_dates, columns=sector_cols)
top = mom[sector_cols].idxmax(axis=1)
for dt, name in top.items():
    weights.loc[dt, name] = 1.0

# Portfolio monthly returns
port_rets = compute_portfolio_returns(weights, monthly_rets[sector_cols])
cum = (1 + port_rets).cumprod()

print('Sharpe:', sharpe_ratio(port_rets))
print('Max DD:', max_drawdown(cum))
cum.plot(title='Baseline Momentum Rotation Equity Curve')
plt.show()

## 7) Model-based strategy (optional)

In [ ]:
if USE_MODEL:
    from sklearn.ensemble import RandomForestRegressor

    # Create target: next month's return for each sector
    target = monthly_rets[sector_cols].shift(-1).loc[features.index]

    X = features.copy()
    preds = pd.DataFrame(index=features.index, columns=sector_cols, dtype=float)

    for i, dt in enumerate(features.index):
        # Rolling train window
        train_end = dt
        train_start = dt - pd.DateOffset(months=ROLLING_TRAIN_WINDOW_MONTHS)

        X_train = X.loc[(X.index >= train_start) & (X.index < train_end)]
        y_train = target.loc[X_train.index]

        if len(X_train) < 24:
            continue

        # Train a separate model per sector
        for s in sector_cols:
            y = y_train[s].dropna()
            Xs = X_train.loc[y.index]
            if len(y) < 24:
                continue
            m = RandomForestRegressor(n_estimators=200, random_state=RANDOM_STATE)
            m.fit(Xs.values, y.values)
            preds.loc[dt, s] = m.predict(X.loc[[dt]].values)[0]

    # Build weights: long top predicted sector
    model_weights = pd.DataFrame(0, index=preds.index, columns=sector_cols)
    top_pred = preds.idxmax(axis=1)
    for dt, s in top_pred.dropna().items():
        model_weights.loc[dt, s] = 1.0

    model_port = compute_portfolio_returns(model_weights, monthly_rets[sector_cols])
    model_cum = (1 + model_port).cumprod()

    print('Model Sharpe:', sharpe_ratio(model_port))
    print('Model Max DD:', max_drawdown(model_cum))

    model_cum.plot(title='Model-based Rotation Equity Curve')
    plt.show()

## 8) Notes / Next steps
- Replace series IDs with your chosen sector index tickers/series.
- Add transaction costs, volatility targeting, and benchmark comparison.
- Consider using ETFs (XLK, XLF, etc.) data rather than FRED indices if desired.


## 9) Download FRED database (for dashboard)

In [ ]:
# Some dashboards expect a local sqlite database built from FRED
# This cell downloads the DB from the repo's releases (if available).

import urllib.request

db_url = 'https://github.com/bigbathtub101/sector-rotation-system/releases/latest/download/fred_data.sqlite'
db_path = repo_root / 'fred_data.sqlite'

try:
    urllib.request.urlretrieve(db_url, db_path)
    print('Database downloaded:', db_path)
except Exception as e:
    print('Could not download DB:', e)
    print('Database downloaded. Place it in the repo root for the dashboard.')